# GHSA benchmark: dataset analysis

This notebook studies the GitHub-reviewed advisory snapshot and the benchmark derived from it, so preprocessing questions stay out of `experiments.ipynb`. It shows how many advisories are usable, how CWE labels are distributed, how much the distribution is concentrated in a few CWEs, which labels the CWE catalog cannot represent, and whether the `dev`/`test` split preserves the label distribution.

Prepare data first from the repository root with `uv run python -m experiment.fetch_sources`. Labels are GitHub curators' `cwe_ids` on reviewed advisories; they are curated and consistently sourced, but not an independent expert adjudication. This notebook does not display advisory text.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

from classification import CWEGraph, normalize_label
from experiment.ghsa import build_benchmark, describe, load_advisories

## Configure paths

Defaults match the layout written by `experiment.fetch_sources`. Run from the repository root.

In [ ]:
DATA = Path.cwd() / "experiment" / "data"
SNAPSHOT = DATA / "raw" / "advisory-database" / "advisories" / "github-reviewed"
PREPARED = DATA / "prepared"
CATALOG = Path.cwd() / "data" / "all.csv"
SEED, DEV_FRACTION = 0, 0.2

## Snapshot provenance and advisory funnel

`advisory_exclusions` counts reviewed-directory files that cannot be used at all (no CWE, withdrawn, unreviewed, missing text). The benchmark exclusions then count advisories dropped because none of their CWEs exist in the catalog, because the catalog cites them as Observed Examples (leakage for description-based scorers), or because alias/text duplicates disagree on labels.

In [ ]:
report = json.loads((PREPARED / "dataset-report.json").read_text(encoding="utf-8"))
catalog = CWEGraph(CATALOG).labels
advisories, advisory_exclusions = load_advisories(SNAPSHOT)
benchmark, audit = build_benchmark(advisories, catalog, seed=SEED, dev_fraction=DEV_FRACTION)
on_disk = [json.loads(line) for line in (PREPARED / "benchmark.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
assert [r["id"] for r in benchmark] == [r["id"] for r in on_disk], "Rebuilt benchmark differs from prepared file; check SEED/DEV_FRACTION/catalog"
{
    "source": report["source"],
    "advisories_with_cwe": len(advisories),
    "advisory_exclusions": advisory_exclusions,
    "benchmark_exclusions": audit["exclusions"],
    "benchmark_records": audit["records"],
    "split_counts": audit["split_counts"],
}

## CWE labels the catalog cannot represent

GitHub sometimes assigns CWE *categories* or deprecated entries. The catalog loader keeps only active weaknesses, so these labels are removed from gold, and advisories with no remaining label are excluded. A large count here means the classifier is structurally unable to produce what curators chose.

In [ ]:
unsupported = Counter(audit["ineligible_labels"])
{"distinct_unsupported_cwes": len(unsupported), "label_occurrences_removed": sum(unsupported.values()), "most_common": unsupported.most_common(15)}

## Label distribution

`top_10_label_share` is the fraction of label occurrences covered by the ten most common CWEs. A high share means overall accuracy is dominated by a handful of classes; use per-label or macro views before claiming broad coverage. `label_abstraction` shows how specific curator labels are (Pillar/Class/Base/Variant), which matters for traversal stopping policies.

In [ ]:
all_labeled = describe(advisories, catalog)
benchmark_summary = describe(benchmark, catalog)
{key: benchmark_summary[key] for key in ("records", "distinct_cwes", "top_10_label_share", "cwes_with_at_most_5_examples", "labels_per_record", "label_abstraction")}

In [ ]:
def name(label):
    return catalog[normalize_label(label)]["Name"]

top = benchmark_summary["top_cwes"]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh([f"{label}: {name(label)[:45]}" for label, _ in reversed(top)], [count for _, count in reversed(top)])
ax.set_xlabel("advisories")
ax.set_title("Most common benchmark CWEs")
fig.tight_layout()

In [ ]:
frequencies = sorted(Counter(l for r in benchmark for l in r["labels"]).values(), reverse=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(frequencies) + 1), frequencies, marker=".")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("CWE rank")
ax.set_ylabel("advisories")
ax.set_title("Rank-frequency of benchmark CWEs (long tail)")
fig.tight_layout()

## Ecosystems and publication years

The benchmark covers open-source package advisories only. These distributions show which ecosystems and periods dominate, and therefore what the metrics can and cannot generalize to.

In [ ]:
{"ecosystems": benchmark_summary["ecosystems"], "published_years": benchmark_summary["published_years"]}

## Split balance

The split is stratified by exact label set with seeded stochastic rounding, so each CWE should appear in `dev` at roughly the configured fraction. Large deviations below indicate rare labels where the split is necessarily noisy; labels present only in `test` cannot be tuned on.

In [ ]:
by_split = {split: Counter(l for r in benchmark if r["split"] == split for l in r["labels"]) for split in ("dev", "test")}
rows = []
for label, count in Counter(l for r in benchmark for l in r["labels"]).most_common():
    rows.append((label, count, by_split["dev"][label] / count))
{
    "top_labels_dev_fraction": [(label, count, round(fraction, 3)) for label, count, fraction in rows[:20]],
    "labels_only_in_test": sum(by_split["dev"][label] == 0 for label, _, _ in rows),
    "labels_only_in_dev": sum(by_split["test"][label] == 0 for label, _, _ in rows),
}